# Diplomado en Ciencia de Datos UC

### Módulo: Modelos Matemáticos y Estadísticos Clásicos
### Curso: Modelos ANOVA y Regresión Lineal
### Profesora: Ana María Alvarado
#### Fecha: 02 de Julio de 2026


In [ ]:
install.packages("skimr")
install.packages('agricolae')
install.packages("performance")
install.packages("see")
install.packages("car")


In [ ]:
install.packages("car")

In [ ]:
install.packages("effects")

In [ ]:
library(tidyverse)
library(skimr)

# 1. Introducción

**¿Cambia el salario de un trabajador según la industria donde se desempeña?**


In [ ]:
# Importamos datos

df_salario <- readxl::read_excel("salarios.xlsx")
# Salario: variable continua
# Industria: variable categórica
# Sexo: Variable categórica
skim(df_salario)


Analizamos el salario por tipo de industria

In [ ]:
# Análisis exploratorio de datos
df_salario %>%
  group_by(Industria) %>%
  skimr::skim(Salario) # función skim del paquete skimr



In [ ]:
# Ajuste de datos
df_salario <- df_salario %>%
  mutate(Industria = factor(Industria, levels = c("Retail", "Banca", "Minería")))

In [ ]:
# Gráfico de boxplot por industria ------------------------------------------------------
df_salario %>% ggplot( aes(x = Industria, y = Salario)) +
  geom_boxplot(fill = "skyblue") +
  labs(title = "Distribución del Salario según industria")+
  theme_minimal(base_size = 15) +
  stat_summary(fun = mean,
               fill = "salmon",
               shape = 23)

**¿Cambia el salario según el sexo del trabajador?**

In [ ]:
# Gráfico de boxplot por sexo ------------------------------------------------------
df_salario %>%
  ggplot(aes(y = Salario, x = Sexo)) +
  geom_boxplot(fill = "lightblue") +
  labs(title = "Efecto del Sexo en el Salario",
       y = "Salario", x = "Sexo") +
  theme_minimal(base_size = 14)+
  stat_summary(fun = mean,
  col = "salmon",
  shape = 23)

**¿Existe interacción entre la variable sexo y tipo de industria?**

In [ ]:
df_salario %>%
  ggplot(aes( y = Salario, x = Sexo)) +
  geom_boxplot(aes(fill = Sexo)) +
  facet_grid(. ~ Industria) +
  labs(title = "Efecto de Sexo e Industria en el Salario",
       y = "Salario", x = "Sexo") +
  theme_minimal(base_size = 14)+
  theme(legend.position = "none")

In [ ]:
df_salario %>%
  group_by(Sexo, Industria) %>%
  summarise(media = mean(Salario, na.rm = TRUE), .groups = "drop_last") %>%
  ggplot(aes(x = Industria, y = media, group = Sexo, color = Sexo)) +
  geom_line(linewidth = 1.2) +
  geom_point(size = 3) +
  labs(title = "Gráfico de Interacción entre Género e Industria?",
       y = "Salario promedio", x = "Industria") +
  theme_minimal(base_size = 14)

# 2. Modelo Anova de un Factor

In [ ]:
modelo_aov1 = aov(Salario ~ Industria, data =df_salario)
modelo_aov1

In [ ]:
modelo_aov2 = aov(Salario ~ Sexo, data =df_salario)
modelo_aov2

In [ ]:
plot(effects::allEffects(modelo_aov1))


# 3. Modelo Anova de dos Factores con Efecto Aditivo




In [ ]:
modelo_aov3 = aov(Salario ~ Industria + Sexo, data =df_salario)
summary(modelo_aov3)

# 4. Modelo Anova de dos Factores con Interacción


In [ ]:
modelo_aov4= aov(Salario ~ Industria*Sexo, data =df_salario)
modelo_aov5= aov(Salario ~ Industria:Sexo, data =df_salario)
modelo_aov6= aov(Salario ~ Industria + Industria:Sexo, data =df_salario)


5. Test de Comparación de Medias

In [ ]:
TukeyHSD(modelo_aov1)
plot(TukeyHSD(modelo_aov1))

In [ ]:
# Realizar el test de Scheffé usando el paquete agricolae
scheffe_test_modelo_aov1 <- agricolae::scheffe.test(modelo_aov1, "Industria",
                                       group=TRUE,
                                     console=TRUE)

scheffe_test_modelo_aov1

# 5. Análisis de los supuestos del modelo

In [ ]:
plot(modelo_aov5)


In [ ]:
# Generar los gráficos de diagnóstico con check_model() del paquete performance
performance::check_model(modelo_aov5)

### Test de Normalidad

H0: Los errores distribuyen normal

H1: Los errores no distribuyen

In [ ]:
# Test de normalidad

shapiro.test(resid(modelo_aov1))
qqnorm(resid(modelo_aov1))
qqline(resid(modelo_aov1), col = "red")

### Test de homocedasticidad

H0: Varianzas iguales

H1: Varianzas distintas

In [ ]:
bartlett.test(Salario ~ Industria, df_salario)


In [ ]:
fligner.test(Salario ~ Industria, data = df_salario)


In [ ]:
car::leveneTest(Salario ~ Industria, data = df_salario)